
# InvariantRRF V44 — Novelty-Strengthening Experiments

This notebook is intentionally **fusion-only**. It does not retrain E5, regenerate the canonical BM25/dense/EnsembleDistil runs, or alter the frozen v4.3 results.

It adds two reviewer-facing audits:

## Experiment A — Plain nested RRF is not a replication-invariance contract
Recent retrieval systems use hierarchical/nested RRF to give a group of related lists one downstream vote. This experiment tests the narrower mathematical question relevant to InvariantRRF:

> If a family already contains multiple non-identical members, can adding an exact copy of one member change a plain nested-RRF output?

We test this on **real families only**:
- the canonical three-member BM25 parameter family on SciFact, FiQA, and ArguAna;
- the two-checkpoint SPLADE family (EnsembleDistil + SelfDistil) on SciFact and ArguAna.

Methods are compared **to their own pre-copy output**:
- flat ordinary RRF;
- plain nested RRF: ordinary RRF within the family, then ordinary RRF across the family consensus and outside sources;
- MC-RRF: exact-copy collapse within declared families followed by fixed family mass.

This is a generic hierarchical-RRF baseline, **not a reproduction of any particular published system**.

## Experiment B — AutoMC provenance-recovery / threshold-transfer audit
The paper currently treats automatic RBO-based family inference as exploratory. This audit tests whether one RBO threshold can recover the declared families across the real BM25 and SPLADE scenarios.

Threshold selection never uses qrels. We report:
- exact partition recovery;
- pairwise same-family precision / recall / F1;
- downstream top-10 agreement with declared MC-RRF;
- leave-one-dataset-out threshold transfer.

If a single threshold works well, say so. If it does not, that supports keeping provenance declared. Do **not** force a desired conclusion.

### Required Kaggle inputs
Attach:
1. the canonical v4.3 output/archive containing `runs/` and `RUN_MANIFEST.json`;
2. the executed SPLADE two-checkpoint closure output/archive containing `generated_runs/*_splade_selfdistil_top500.json`;
3. benchmark folders only if you also want optional nDCG drift columns. Structural invariance metrics do not require qrels.


In [ ]:

from pathlib import Path
from collections import defaultdict
from itertools import combinations
import hashlib, json, math, shutil, zipfile

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()

OUT = ROOT / "InvariantRRF_V44_Novelty_Strengthening"
if OUT.exists():
    shutil.rmtree(OUT)
OUT.mkdir(parents=True, exist_ok=True)

RRF_K = 60.0
TOP_K = 10
DEPTHS = [20, 50, 100]
PRIMARY_DEPTH = 50
RBO_P = 0.90

# Optional manual overrides. Leave None for auto-discovery.
CANONICAL_ZIP = None
CANONICAL_ROOT = None
SPLADE_CLOSURE_ZIP = None
SPLADE_CLOSURE_ROOT = None

print("INPUT_ROOT:", INPUT_ROOT)
print("OUT:", OUT)


In [ ]:

# ============================================================
# 1. Locate / verify frozen inputs
# ============================================================

def sha256_file(path, chunk=1024 * 1024):
    path = Path(path)
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def find_canonical_root(root):
    root = Path(root)
    if (root / "runs").is_dir():
        return root
    for m in sorted(root.rglob("RUN_MANIFEST.json")):
        if (m.parent / "runs").is_dir():
            return m.parent
    for d in sorted(p for p in root.rglob("runs") if p.is_dir()):
        if any(d.glob("*_dense_STRICT_top1000.json")):
            return d.parent
    raise FileNotFoundError(f"No canonical root with runs/ found under {root}")

def discover_zip(keywords):
    zips = list(INPUT_ROOT.rglob("*.zip")) if INPUT_ROOT.exists() else []
    scored = []
    for p in zips:
        n = p.name.lower()
        score = sum(1 for k in keywords if k.lower() in n)
        if score:
            scored.append((score, len(n), str(p), p))
    if not scored:
        return None
    scored.sort(reverse=True)
    return scored[0][-1]

# Canonical
if CANONICAL_ROOT:
    CANON = find_canonical_root(Path(CANONICAL_ROOT))
else:
    z = Path(CANONICAL_ZIP) if CANONICAL_ZIP else discover_zip(
        ["invariantrrf", "canonical", "deepdense", "taskadaptivek"]
    )
    if z is not None and z.exists():
        extract = ROOT / "_v44_canonical_import"
        if extract.exists():
            shutil.rmtree(extract)
        extract.mkdir(parents=True)
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(extract)
        CANON = find_canonical_root(extract)
        print("Canonical archive:", z)
    else:
        CANON = find_canonical_root(INPUT_ROOT)

RUNS = CANON / "runs"
TABLES = CANON / "tables"
STATS = CANON / "statistics"

manifest_path = CANON / "RUN_MANIFEST.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text("utf-8"))
    failures = []
    for rel, expected in manifest.get("files", {}).items():
        p = CANON / rel
        if not p.exists():
            failures.append((rel, "MISSING"))
        elif sha256_file(p) != expected:
            failures.append((rel, "HASH_MISMATCH"))
    if failures:
        raise AssertionError(f"Canonical manifest failures: {failures[:10]}")
    print("Canonical manifest: PASS", len(manifest.get("files", {})), "files")
else:
    print("WARNING: canonical RUN_MANIFEST.json not found")

# SPLADE closure
def find_splade_closure_root(root):
    root = Path(root)
    candidates = []
    if (root / "generated_runs").is_dir():
        candidates.append(root)
    candidates += [p.parent for p in root.rglob("SPLADE_V43_CLOSURE_PROTOCOL.json")]
    candidates += [p.parent for p in root.rglob("CLOSURE_MANIFEST.json")
                   if (p.parent / "generated_runs").is_dir()]
    for c in candidates:
        if (c / "generated_runs" / "SciFact_splade_selfdistil_top500.json").exists():
            return c
    raise FileNotFoundError("Could not locate executed SPLADE closure output")

if SPLADE_CLOSURE_ROOT:
    SPLADE_CLOSURE = find_splade_closure_root(Path(SPLADE_CLOSURE_ROOT))
else:
    z2 = Path(SPLADE_CLOSURE_ZIP) if SPLADE_CLOSURE_ZIP else discover_zip(
        ["splade", "twocheckpoint", "closure"]
    )
    if z2 is not None and z2.exists():
        extract2 = ROOT / "_v44_splade_closure_import"
        if extract2.exists():
            shutil.rmtree(extract2)
        extract2.mkdir(parents=True)
        with zipfile.ZipFile(z2, "r") as zf:
            zf.extractall(extract2)
        SPLADE_CLOSURE = find_splade_closure_root(extract2)
        print("SPLADE closure archive:", z2)
    else:
        SPLADE_CLOSURE = find_splade_closure_root(INPUT_ROOT)

SELF_RUNS = SPLADE_CLOSURE / "generated_runs"

print("CANON:", CANON)
print("SPLADE_CLOSURE:", SPLADE_CLOSURE)
assert RUNS.is_dir()
assert SELF_RUNS.is_dir()


In [ ]:

# ============================================================
# 2. Load ranked runs and optional qrels
# ============================================================

def load_run(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    raw = json.loads(path.read_text("utf-8"))
    out = {}
    for qid, seq in raw.items():
        norm = []
        for x in seq:
            if isinstance(x, (list, tuple)):
                norm.append((str(x[0]), float(x[1]) if len(x) > 1 else 0.0))
            else:
                norm.append((str(x), 0.0))
        ids = [d for d, _ in norm]
        if len(ids) != len(set(ids)):
            raise AssertionError(f"Duplicate document IDs in {path.name}/{qid}")
        out[str(qid)] = norm
    return out

def run_path(ds, source):
    mapping = {
        "bm25": RUNS / f"{ds}_bm25_top1000.json",
        "bm25_lowb": RUNS / f"{ds}_bm25_lowb_top1000.json",
        "bm25_highb": RUNS / f"{ds}_bm25_highb_top1000.json",
        "dense": RUNS / f"{ds}_dense_STRICT_top1000.json",
        "splade_ensemble": RUNS / f"{ds}_splade_ensemble_top500.json",
        "splade_self": SELF_RUNS / f"{ds}_splade_selfdistil_top500.json",
    }
    return mapping[source]

RUN_CACHE = {}
def get_run(ds, source):
    key = (ds, source)
    if key not in RUN_CACHE:
        RUN_CACHE[key] = load_run(run_path(ds, source))
    return RUN_CACHE[key]

# Optional qrels. Structural metrics do not depend on this.
DATASET_DIRNAMES = {
    "SciFact": "scifact_mpdr",
    "FiQA": "fiqa_mpdr",
    "ArguAna": "arguana_mpdr",
}

def find_dataset_root(dirname):
    if not INPUT_ROOT.exists():
        return None
    for p in INPUT_ROOT.rglob(dirname):
        if p.is_dir() and (p / "dev" / "qrels.tsv").exists():
            return p
    return None

def load_qrels(root):
    if root is None:
        return {}
    path = Path(root) / "dev" / "qrels.tsv"
    out = defaultdict(dict)
    for line in path.read_text("utf-8").splitlines():
        sp = line.split("\t")
        if len(sp) < 2:
            continue
        qid, did = str(sp[0]), str(sp[1])
        try:
            rel = float(sp[2]) if len(sp) >= 3 and sp[2] != "" else 1.0
        except Exception:
            rel = 1.0
        if rel > 0:
            out[qid][did] = rel
    return dict(out)

QRELS = {}
for ds, dirname in DATASET_DIRNAMES.items():
    root = find_dataset_root(dirname)
    QRELS[ds] = load_qrels(root)
    print(ds, "qrels:", len(QRELS[ds]), "root:", root)


In [ ]:

# ============================================================
# 3. Fusion / evaluation helpers
# ============================================================

def prefix_docs(seq, depth):
    return [str(d) for d, _ in seq[:min(int(depth), len(seq))]]

def rrf_fuse(rankings, depth, k=RRF_K, weights=None):
    weights = weights or {name: 1.0 for name in rankings}
    scores = defaultdict(float)
    for name, seq in rankings.items():
        w = float(weights[name])
        for r, did in enumerate(prefix_docs(seq, depth), start=1):
            scores[did] += w / (float(k) + r)
    return sorted(scores.items(), key=lambda x: (-x[1], x[0]))

def signature(seq):
    ids = [str(d) for d, _ in seq]
    return hashlib.sha256("\x1f".join(ids).encode("utf-8")).hexdigest()

def mc_rrf(rankings, family_map, depth, k=RRF_K):
    # Exact-copy collapse WITHIN declared families, then equal split of one mass unit
    # across unique members of each family.
    fams = defaultdict(list)
    for name in rankings:
        fams[str(family_map[name])].append(name)

    reps = {}
    rep_family = {}
    for fam, members in fams.items():
        by_sig = defaultdict(list)
        for m in members:
            by_sig[signature(rankings[m])].append(m)
        for _, same in sorted(by_sig.items(), key=lambda kv: tuple(sorted(kv[1]))):
            rep = sorted(same)[0]
            key = f"{fam}::{rep}"
            reps[key] = rankings[rep]
            rep_family[key] = fam

    fam_rep_members = defaultdict(list)
    for rep, fam in rep_family.items():
        fam_rep_members[fam].append(rep)

    weights = {}
    for fam, members in fam_rep_members.items():
        for m in members:
            weights[m] = 1.0 / len(members)

    return rrf_fuse(reps, depth=depth, k=k, weights=weights)

def nested_rrf(family_rankings, outside_rankings, depth, k=RRF_K):
    # Plain two-level RRF:
    # 1) ordinary RRF inside the family;
    # 2) retain top `depth` family-consensus documents;
    # 3) ordinary RRF over that family consensus + each outside source.
    inner = rrf_fuse(family_rankings, depth=depth, k=k)
    inner_seq = [(d, s) for d, s in inner[:depth]]
    outer = {"FAMILY_CONSENSUS": inner_seq, **outside_rankings}
    return rrf_fuse(outer, depth=depth, k=k)

def ndcg_at_k(docids_, rels, k=10):
    if not rels:
        return np.nan
    dcg = 0.0
    for i, d in enumerate(docids_[:k], start=1):
        rel = float(rels.get(str(d), 0.0))
        dcg += (2.0**rel - 1.0) / math.log2(i + 1.0)
    ideal = sorted((float(v) for v in rels.values()), reverse=True)[:k]
    if not ideal:
        return 0.0
    idcg = sum((2.0**rel - 1.0) / math.log2(i + 2.0) for i, rel in enumerate(ideal))
    return dcg / idcg if idcg > 0 else 0.0

def top_docs(res, k=TOP_K):
    return [d for d, _ in res[:k]]

def rbo_finite(a, b, p=RBO_P, depth=None):
    a = list(a); b = list(b)
    depth = min(depth or max(len(a), len(b)), max(len(a), len(b)))
    if depth <= 0:
        return 1.0
    A, B = set(), set()
    num = den = 0.0
    for d in range(1, depth + 1):
        if d <= len(a): A.add(a[d-1])
        if d <= len(b): B.add(b[d-1])
        w = p ** (d - 1)
        num += (len(A & B) / d) * w
        den += w
    return num / den if den else 1.0

def partition_pair_metrics(true_map, pred_map):
    names = sorted(true_map)
    tp = fp = fn = tn = 0
    exact = True
    for a, b in combinations(names, 2):
        yt = true_map[a] == true_map[b]
        yp = pred_map[a] == pred_map[b]
        exact &= (yt == yp)
        if yt and yp: tp += 1
        elif (not yt) and yp: fp += 1
        elif yt and (not yp): fn += 1
        else: tn += 1
    precision = tp / (tp + fp) if (tp + fp) else (1.0 if (tp + fn) == 0 else 0.0)
    recall = tp / (tp + fn) if (tp + fn) else 1.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0
    return dict(exact_partition=int(exact), pair_precision=precision, pair_recall=recall, pair_f1=f1)

def infer_families_rbo(rankings, depth, threshold, p=RBO_P):
    names = sorted(rankings)
    parent = {n: n for n in names}

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for a, b in combinations(names, 2):
        ra = prefix_docs(rankings[a], depth)
        rb = prefix_docs(rankings[b], depth)
        if rbo_finite(ra, rb, p=p, depth=depth) >= threshold:
            union(a, b)

    roots = {}
    out = {}
    for n in names:
        r = find(n)
        if r not in roots:
            roots[r] = f"AutoFamily-{len(roots)+1}"
        out[n] = roots[r]
    return out



## Experiment A — replication inside an already heterogeneous family

The important distinction is **method-to-itself invariance**.

For each query and method:
1. compute the method's fused top-10 from the real family;
2. add an exact copy of one existing family member;
3. recompute the same method;
4. test whether its order/set/top-1 changed.

MC-RRF is asserted to remain exactly invariant. Flat ordinary RRF and plain nested RRF are allowed to change.


In [ ]:

# ============================================================
# 4. Experiment A: plain nested RRF vs MC-RRF
# ============================================================

SCENARIOS = []

# Real BM25 parameter family.
for ds in ["SciFact", "FiQA", "ArguAna"]:
    fam = ["bm25", "bm25_lowb", "bm25_highb"]
    outside = ["dense"] + (["splade_ensemble"] if (RUNS / f"{ds}_splade_ensemble_top500.json").exists() else [])
    SCENARIOS.append({
        "scenario": "BM25-family",
        "dataset": ds,
        "family": fam,
        "outside": outside,
    })

# Real SPLADE checkpoint family.
for ds in ["SciFact", "ArguAna"]:
    SCENARIOS.append({
        "scenario": "SPLADE-checkpoint-family",
        "dataset": ds,
        "family": ["splade_ensemble", "splade_self"],
        "outside": ["bm25"],
    })

rep_rows = []

for sc in SCENARIOS:
    ds = sc["dataset"]
    fam_names = sc["family"]
    outside_names = sc["outside"]

    source_runs = {s: get_run(ds, s) for s in fam_names + outside_names}
    qsets = [set(r.keys()) for r in source_runs.values()]
    qids = sorted(set.intersection(*qsets))
    assert qids, sc

    print("\n", ds, sc["scenario"], "queries=", len(qids),
          "family=", fam_names, "outside=", outside_names)

    for depth in DEPTHS:
        for copied in fam_names:
            for q in qids:
                family_base = {n: source_runs[n][q] for n in fam_names}
                outside = {n: source_runs[n][q] for n in outside_names}

                # Add an exact represented copy.
                copied_name = copied + "__EXACT_COPY"
                family_attacked = {**family_base, copied_name: family_base[copied]}

                # FLAT ordinary
                flat_base = rrf_fuse({**family_base, **outside}, depth=depth)
                flat_att = rrf_fuse({**family_attacked, **outside}, depth=depth)

                # NESTED ordinary
                nested_base = nested_rrf(family_base, outside, depth=depth)
                nested_att = nested_rrf(family_attacked, outside, depth=depth)

                # MC-RRF
                base_all = {**family_base, **outside}
                att_all = {**family_attacked, **outside}
                base_map = {n: "TARGET_FAMILY" for n in fam_names}
                base_map.update({n: f"OUTSIDE::{n}" for n in outside_names})
                att_map = dict(base_map)
                att_map[copied_name] = "TARGET_FAMILY"

                mc_base = mc_rrf(base_all, base_map, depth=depth)
                mc_att = mc_rrf(att_all, att_map, depth=depth)

                method_pairs = {
                    "Flat ordinary RRF": (flat_base, flat_att),
                    "Plain nested RRF": (nested_base, nested_att),
                    "MC-RRF": (mc_base, mc_att),
                }

                rels = QRELS.get(ds, {}).get(q, {})
                for method, (base_res, att_res) in method_pairs.items():
                    b = top_docs(base_res)
                    a = top_docs(att_res)
                    nb = ndcg_at_k(b, rels, TOP_K)
                    na = ndcg_at_k(a, rels, TOP_K)
                    rep_rows.append({
                        "scenario": sc["scenario"],
                        "dataset": ds,
                        "qid": q,
                        "depth": depth,
                        "copied_member": copied,
                        "method": method,
                        "order_preserved": int(a == b),
                        "set_preserved": int(set(a) == set(b)),
                        "top1_preserved": int(bool(a) and bool(b) and a[0] == b[0]),
                        "abs_ndcg_drift": abs(na - nb) if np.isfinite(nb) and np.isfinite(na) else np.nan,
                    })

rep_perq = pd.DataFrame(rep_rows)
rep_summary = (
    rep_perq
    .groupby(["scenario", "dataset", "depth", "copied_member", "method"], as_index=False)
    .agg(
        n=("qid", "count"),
        order_preservation=("order_preserved", "mean"),
        set_preservation=("set_preserved", "mean"),
        top1_preservation=("top1_preserved", "mean"),
        mean_abs_ndcg_drift=("abs_ndcg_drift", "mean"),
    )
)

rep_perq.to_csv(OUT / "nested_rrf_replication_per_query.csv", index=False)
rep_summary.to_csv(OUT / "nested_rrf_replication_summary.csv", index=False)

# Formal MC invariance check.
mc = rep_summary[rep_summary.method == "MC-RRF"]
assert np.allclose(mc.order_preservation, 1.0)
assert np.allclose(mc.set_preservation, 1.0)
assert np.allclose(mc.top1_preservation, 1.0)
if mc.mean_abs_ndcg_drift.notna().any():
    assert np.nanmax(np.abs(mc.mean_abs_ndcg_drift.to_numpy(float))) <= 1e-15

print("MC-RRF exact-replication invariance: PASS")
display(rep_summary[rep_summary.depth == PRIMARY_DEPTH])



### Paper-facing interpretation rule for Experiment A

Use this experiment only to support the following narrow statement if the data show it:

> Plain hierarchical pre-aggregation by ordinary RRF does not itself guarantee source-replication invariance once a family contains non-identical members.

Do **not** claim that a particular published nested-RRF system is invalid or inferior. The experiment tests a mathematical property of the generic two-level construction.



## Experiment B — AutoMC provenance recovery and threshold transfer

This experiment does **not** select thresholds by relevance effectiveness.

For each real-family scenario:
- compute query-level pairwise finite-prefix RBO;
- infer connected components at each threshold;
- compare the inferred partition with the declared provenance partition;
- compare AutoMC's fused top-10 with declared MC-RRF.

Then perform leave-one-dataset-out threshold selection using **provenance-recovery F1 only**.


In [ ]:

# ============================================================
# 5. Experiment B: AutoMC threshold audit
# ============================================================

THRESHOLDS = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.93, 0.95, 0.97, 0.99]
AUTO_DEPTHS = [50, 100]

auto_rows = []

for sc in SCENARIOS:
    ds = sc["dataset"]
    fam_names = sc["family"]
    outside_names = sc["outside"]

    source_names = fam_names + outside_names
    source_runs = {s: get_run(ds, s) for s in source_names}
    qids = sorted(set.intersection(*(set(r) for r in source_runs.values())))

    true_map = {n: "TARGET_FAMILY" for n in fam_names}
    true_map.update({n: f"OUTSIDE::{n}" for n in outside_names})

    for depth in AUTO_DEPTHS:
        for q in qids:
            rankings = {n: source_runs[n][q] for n in source_names}
            declared_res = mc_rrf(rankings, true_map, depth=depth)
            declared_top = top_docs(declared_res)
            rels = QRELS.get(ds, {}).get(q, {})
            declared_ndcg = ndcg_at_k(declared_top, rels, TOP_K)

            for threshold in THRESHOLDS:
                pred_map = infer_families_rbo(rankings, depth=depth, threshold=threshold)
                pm = partition_pair_metrics(true_map, pred_map)

                auto_res = mc_rrf(rankings, pred_map, depth=depth)
                auto_top = top_docs(auto_res)
                auto_ndcg = ndcg_at_k(auto_top, rels, TOP_K)

                auto_rows.append({
                    "scenario": sc["scenario"],
                    "dataset": ds,
                    "qid": q,
                    "depth": depth,
                    "threshold": threshold,
                    **pm,
                    "order_agreement_with_declared_mc": int(auto_top == declared_top),
                    "set_agreement_with_declared_mc": int(set(auto_top) == set(declared_top)),
                    "abs_ndcg_difference_from_declared_mc": (
                        abs(auto_ndcg - declared_ndcg)
                        if np.isfinite(auto_ndcg) and np.isfinite(declared_ndcg)
                        else np.nan
                    ),
                })

auto_perq = pd.DataFrame(auto_rows)

auto_summary = (
    auto_perq
    .groupby(["scenario", "dataset", "depth", "threshold"], as_index=False)
    .agg(
        n=("qid", "count"),
        exact_partition_recovery=("exact_partition", "mean"),
        pair_precision=("pair_precision", "mean"),
        pair_recall=("pair_recall", "mean"),
        pair_f1=("pair_f1", "mean"),
        order_agreement_with_declared_mc=("order_agreement_with_declared_mc", "mean"),
        set_agreement_with_declared_mc=("set_agreement_with_declared_mc", "mean"),
        mean_abs_ndcg_difference_from_declared_mc=("abs_ndcg_difference_from_declared_mc", "mean"),
    )
)

auto_perq.to_csv(OUT / "automc_provenance_per_query.csv", index=False)
auto_summary.to_csv(OUT / "automc_provenance_summary.csv", index=False)

display(auto_summary[auto_summary.depth == PRIMARY_DEPTH].head(30))


In [ ]:

# ============================================================
# 6. Global and leave-one-dataset-out threshold transfer
# ============================================================

def choose_threshold(summary_rows):
    # qrels-free selection:
    # 1) maximize macro exact partition recovery
    # 2) maximize macro pair F1
    # 3) closest to 0.75 only as deterministic final tie-break
    g = (
        summary_rows.groupby("threshold", as_index=False)
        .agg(
            exact=("exact_partition_recovery", "mean"),
            f1=("pair_f1", "mean"),
        )
    )
    g["tie_distance"] = (g["threshold"] - 0.75).abs()
    g = g.sort_values(["exact", "f1", "tie_distance", "threshold"],
                      ascending=[False, False, True, True])
    return float(g.iloc[0]["threshold"]), g

primary = auto_summary[auto_summary.depth == PRIMARY_DEPTH].copy()

global_thr, global_grid = choose_threshold(primary)
print("GLOBAL qrels-free provenance threshold:", global_thr)
display(global_grid)

lodo_rows = []
datasets = sorted(primary.dataset.unique())

for held in datasets:
    train = primary[primary.dataset != held]
    test = primary[primary.dataset == held]
    thr, _ = choose_threshold(train)
    chosen = test[np.isclose(test.threshold, thr)]
    for r in chosen.itertuples():
        lodo_rows.append({
            "held_out_dataset": held,
            "scenario": r.scenario,
            "selected_threshold_from_other_datasets": thr,
            "exact_partition_recovery": r.exact_partition_recovery,
            "pair_f1": r.pair_f1,
            "set_agreement_with_declared_mc": r.set_agreement_with_declared_mc,
            "mean_abs_ndcg_difference_from_declared_mc": r.mean_abs_ndcg_difference_from_declared_mc,
        })

lodo_df = pd.DataFrame(lodo_rows)
lodo_df.to_csv(OUT / "automc_leave_one_dataset_out.csv", index=False)

# Per-dataset oracle threshold, still qrels-free, only for diagnosis.
oracle_rows = []
for (scenario, ds), g in primary.groupby(["scenario", "dataset"]):
    thr, grid = choose_threshold(g)
    best = grid[np.isclose(grid.threshold, thr)].iloc[0]
    oracle_rows.append({
        "scenario": scenario,
        "dataset": ds,
        "oracle_qrels_free_threshold": thr,
        "exact_partition_recovery": best["exact"],
        "pair_f1": best["f1"],
    })
oracle_df = pd.DataFrame(oracle_rows)
oracle_df.to_csv(OUT / "automc_per_dataset_oracle_thresholds.csv", index=False)

print("\nLeave-one-dataset-out transfer")
display(lodo_df)

print("\nPer-dataset qrels-free oracle thresholds")
display(oracle_df)


In [ ]:

# ============================================================
# 7. Conservative manuscript decision report
# ============================================================

primary_rep = rep_summary[rep_summary.depth == PRIMARY_DEPTH].copy()

nested = primary_rep[primary_rep.method == "Plain nested RRF"]
nested_noninv_order = int((nested.order_preservation < 1.0 - 1e-15).sum())
nested_noninv_set = int((nested.set_preservation < 1.0 - 1e-15).sum())

global_primary = primary[np.isclose(primary.threshold, global_thr)]
macro_global_exact = float(global_primary.exact_partition_recovery.mean())
macro_global_f1 = float(global_primary.pair_f1.mean())
macro_global_set_agree = float(global_primary.set_agreement_with_declared_mc.mean())

oracle_unique_thresholds = sorted(oracle_df.oracle_qrels_free_threshold.unique().tolist())

report = []
report += [
    "# InvariantRRF V44 novelty-strengthening audit",
    "",
    "## Experiment A: generic plain nested RRF",
    f"- Primary depth: {PRIMARY_DEPTH}",
    f"- Nested-RRF conditions with non-perfect ordered replication preservation: {nested_noninv_order}/{len(nested)}",
    f"- Nested-RRF conditions with non-perfect set replication preservation: {nested_noninv_set}/{len(nested)}",
    "- MC-RRF exact replication invariance: PASS in every tested real-family condition.",
    "",
]

if nested_noninv_order > 0 or nested_noninv_set > 0:
    report += [
        "**Supported narrow interpretation:** plain ordinary-RRF pre-aggregation does not by itself "
        "guarantee replication invariance once the inner family already contains non-identical members.",
        "",
    ]
else:
    report += [
        "**Do not add a strong nested-RRF claim:** the tested real-family conditions happened to remain "
        "stable under the chosen exact-copy attacks. The algebraic counterexample in the paper can remain, "
        "but this benchmark audit does not add empirical support.",
        "",
    ]

report += [
    "## Experiment B: AutoMC provenance recovery",
    f"- Global qrels-free threshold selected from declared-family recovery: {global_thr:.2f}",
    f"- Macro exact partition recovery at that threshold: {macro_global_exact:.3f}",
    f"- Macro pairwise family F1 at that threshold: {macro_global_f1:.3f}",
    f"- Macro top-10 set agreement with declared MC-RRF: {macro_global_set_agree:.3f}",
    f"- Per-dataset/scenario oracle thresholds observed: {oracle_unique_thresholds}",
    "",
]

if macro_global_exact >= 0.95 and len(oracle_unique_thresholds) <= 2:
    report += [
        "**Interpretation:** automatic RBO grouping is more stable than the current manuscript wording suggests "
        "on these controlled real-family scenarios. Do not claim that the threshold is strongly dataset dependent. "
        "Present AutoMC as a promising exploratory approximation, while keeping the theorem conditional on declared provenance.",
    ]
elif macro_global_exact < 0.80 or len(oracle_unique_thresholds) >= 3:
    report += [
        "**Interpretation:** one overlap threshold does not recover declared provenance reliably across the tested "
        "real-family scenarios. This supports keeping AutoMC exploratory and provenance declared.",
    ]
else:
    report += [
        "**Interpretation:** AutoMC is partially transferable but imperfect. Use neutral wording: similarity-based "
        "grouping can approximate declared provenance in some conditions, but the invariant guarantee still depends "
        "on the supplied family map.",
    ]

report_text = "\n".join(report)
(OUT / "MANUSCRIPT_DECISION.md").write_text(report_text, encoding="utf-8")

display(Markdown(report_text))


In [ ]:

# ============================================================
# 8. Simple publication-ready diagnostic figures
# ============================================================

import matplotlib.pyplot as plt

# Figure A: primary-depth replication preservation by method.
fig_df = (
    primary_rep
    .groupby(["method"], as_index=False)
    .agg(order_preservation=("order_preservation", "mean"),
         set_preservation=("set_preservation", "mean"))
)

x = np.arange(len(fig_df))
w = 0.36
fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.bar(x - w/2, fig_df["order_preservation"], width=w, label="Ordered top-10")
ax.bar(x + w/2, fig_df["set_preservation"], width=w, label="Top-10 set")
ax.set_xticks(x)
ax.set_xticklabels(fig_df["method"], rotation=10, ha="right")
ax.set_ylim(0, 1.05)
ax.set_ylabel("Replication preservation")
ax.set_title("Within-family exact-copy audit at depth 50")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "Figure_NestedRRF_ReplicationAudit.pdf", bbox_inches="tight")
fig.savefig(OUT / "Figure_NestedRRF_ReplicationAudit.png", dpi=300, bbox_inches="tight")
plt.show()

# Figure B: AutoMC threshold transfer, macro across scenario/dataset units.
curve = (
    primary
    .groupby("threshold", as_index=False)
    .agg(exact_partition_recovery=("exact_partition_recovery", "mean"),
         pair_f1=("pair_f1", "mean"),
         set_agreement=("set_agreement_with_declared_mc", "mean"))
)

fig, ax = plt.subplots(figsize=(7.2, 4.4))
ax.plot(curve["threshold"], curve["exact_partition_recovery"], marker="o", label="Exact partition recovery")
ax.plot(curve["threshold"], curve["pair_f1"], marker="s", label="Pairwise family F1")
ax.plot(curve["threshold"], curve["set_agreement"], marker="^", label="Top-10 set agreement vs declared MC")
ax.axvline(global_thr, linestyle="--", linewidth=1, label=f"Selected threshold = {global_thr:.2f}")
ax.set_ylim(0, 1.05)
ax.set_xlabel("RBO grouping threshold")
ax.set_ylabel("Rate")
ax.set_title("AutoMC provenance-threshold audit at depth 50")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT / "Figure_AutoMC_ThresholdAudit.pdf", bbox_inches="tight")
fig.savefig(OUT / "Figure_AutoMC_ThresholdAudit.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# 9. Package outputs
# ============================================================

manifest = {}
for p in sorted(OUT.iterdir()):
    if p.is_file():
        manifest[p.name] = sha256_file(p)

(OUT / "OUTPUT_SHA256.json").write_text(
    json.dumps(manifest, indent=2, sort_keys=True),
    encoding="utf-8",
)

zip_path = ROOT / "InvariantRRF_V44_Novelty_Strengthening_Results.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT.iterdir()):
        if p.is_file():
            zf.write(p, arcname=p.name)

print("FINAL OUTPUT DIR:", OUT)
print("FINAL ZIP:", zip_path)
print("Files:")
for p in sorted(OUT.iterdir()):
    print(" -", p.name)
